# Atelier : évaluer l'affinage du service client

Le fil rouge de la formation : le service client de Biclou & Cie, dont le modèle doit trier chaque message (catégorie, priorité) et y répondre selon la charte, en un objet JSON. Le TP de mesure a fixé la barre, le modèle de base avec le meilleur prompt ; le TP d'entraînement a produit un adaptateur QLoRA de Qwen3-1.7B. Cet atelier mesure ce que l'adaptateur vaut : sur la tâche, pour ce que les métriques ne voient pas, et pour ce que le modèle savait faire avant.

1. Recharger le modèle affiné
2. ROUGE-L et perplexité
3. Un LLM juge
4. Calibrer le juge
5. Le biais de position
6. Non-régression : un benchmark
7. Non-régression : hors du service client
8. L'analyse d'erreurs

L'adaptateur est le vôtre si vous l'avez publié à la fin du TP d'entraînement (le secret Colab `HF_TOKEN`), sinon un adaptateur de référence, entraîné par le même code.

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*, et il passe alors en entier en moins d'une heure. Sans GPU, il tourne quand même, en plus petit : Qwen3-0.6B et son adaptateur au lieu de Qwen3-1.7B, sur 4 tickets de test au lieu de 150 (`test_part`), et Qwen3-1.7B pour juge au lieu de Qwen3-4B-Instruct-2507 (Qwen3-0.6B recopie l'exemple de sa consigne au lieu de juger) : d'une à dix minutes par exercice, le premier et ceux du juge étant les plus longs (une minute par verdict). Sur 4 tickets, les chiffres ne disent presque rien : c'est la méthode qui compte.

## Données, modèles et mesures

`test.jsonl` : les 150 tickets de test validés par l'équipe, avec la réponse attendue. `charte.md` : la charte de réponse du service client. `tickets.csv` : l'export brut de l'outil de support, où chaque ticket de test a aussi la réponse qu'un conseiller y a faite. `lm_eval` : lm-evaluation-harness, d'EleutherAI, pour l'exercice 6.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/test.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/charte.md
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/tickets.csv
%pip install -q peft bitsandbytes evaluate rouge_score lm_eval
%pip uninstall -q -y torchao  # la version de Colab est trop ancienne pour peft

In [ ]:
import contextlib
import copy
import gc
import json
import math
import os
import re
import time
import urllib.request

import evaluate
import pandas as pd
import torch
from huggingface_hub import HfApi
from lm_eval import simple_evaluate
from lm_eval.models.huggingface import HFLM
from peft import PeftModel
from transformers import (
  AutoModelForCausalLM,
  AutoTokenizer,
  BitsAndBytesConfig,
  DynamicCache,
)

GPU = torch.cuda.is_available()
# bfloat16 à partir des GPU Ampere (A100, L4, H100) ; le T4 calcule en float16.
BF16 = GPU and torch.cuda.get_device_capability()[0] >= 8
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = (torch.bfloat16 if BF16 else torch.float16) if GPU else torch.float32
DEVICE = "cuda" if GPU else "cpu"
MODELE = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
JUGE = "Qwen/Qwen3-4B-Instruct-2507" if GPU else "Qwen/Qwen3-1.7B"
print(MODELE, "sur GPU" if GPU else "sans GPU, sur 4 tickets")

tokenizer = AutoTokenizer.from_pretrained(MODELE)
tokenizer.padding_side = "left"
test = pd.read_json("test.jsonl", lines=True)
test_part = test[: None if GPU else 4]
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()
tickets = pd.read_csv("tickets.csv").drop_duplicates("ticket_id").set_index("ticket_id")
print(len(test), "tickets de test ;", len(test_part), "mesurés ici")


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()

L'adaptateur, dans le dossier `ADAPTATEUR` : le vôtre, s'il est sur votre Hub et a été entraîné sur `MODELE`, sinon celui de référence, de [shuuchuu/models](https://github.com/shuuchuu/models).

In [ ]:
ADAPTATEUR = "support-sft-lora"
REFERENCE = "https://github.com/shuuchuu/models/raw/refs/heads/main/" + (
  "support-sft-lora-qwen3-1.7b" if GPU else "support-sft-lora-qwen3-0.6b"
)


def adaptateur_du_hub() -> str | None:
  """Télécharge votre adaptateur dans ADAPTATEUR, s'il existe pour MODELE."""
  jeton = os.environ.get("HF_TOKEN")
  try:
    from google.colab import userdata

    jeton = userdata.get("HF_TOKEN")
  except Exception:
    pass
  if not jeton:
    return None
  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/support-sft-lora"
  if not api.repo_exists(depot):
    return None
  with open(api.hf_hub_download(depot, "adapter_config.json")) as f:
    if json.load(f)["base_model_name_or_path"] != MODELE:
      return None
  api.snapshot_download(depot, local_dir=ADAPTATEUR)
  return depot


depot = adaptateur_du_hub()
if depot is None:
  os.makedirs(ADAPTATEUR, exist_ok=True)
  for fichier in ["adapter_config.json", "adapter_model.safetensors"]:
    urllib.request.urlretrieve(f"{REFERENCE}/{fichier}", f"{ADAPTATEUR}/{fichier}")
print("Adaptateur :", depot or REFERENCE)

## La tâche et sa mesure

Le code du TP de mesure, repris tel quel : `CONSIGNES`, la consigne courte que le modèle affiné a reçue à l'entraînement ; `evalue(sorties)`, qui mesure les sorties sur `test_part` (`json_valide`, `categorie`, `priorite`, `charte` pour les règles vérifiables de la charte, `mots`) ; `genere_tout(modele, conversations)`, qui génère par lots, sans échantillonnage ; et les conversations des approches : `conversation_seule(message)` (la consigne seule) et `conversation_charte(message)` (la consigne, la charte complète et trois exemples, le meilleur prompt).

In [ ]:
CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
CONSIGNES = f"""Tu es l'assistant du service client de Biclou & Cie, magasins de vélos.
Pour le message du client, réponds uniquement par un objet JSON avec trois clés :
- categorie : une valeur parmi {", ".join(CATEGORIES)} ;
- priorite : basse, normale ou haute ;
- reponse : la réponse à envoyer au client."""
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)


def lit_json(texte: str) -> dict:
  """L'objet JSON d'une réponse du modèle, ou {} s'il n'est pas valide."""
  texte = texte.strip().removeprefix("```json").removesuffix("```").strip()
  try:
    objet = json.loads(texte)
  except json.JSONDecodeError:
    return {}
  return objet if isinstance(objet, dict) else {}


def respecte_charte(reponse: str) -> bool:
  reponse = reponse.strip()
  return (
    reponse.startswith("Bonjour")
    and reponse.endswith(SIGNATURE)
    and not TUTOIEMENT.search(reponse)
    and 40 <= len(reponse.split()) <= 120
  )


def evalue(sorties: list[str]) -> pd.Series:
  objets = [lit_json(s) for s in sorties]
  reponses = [str(o.get("reponse", "")) for o in objets]
  return (
    pd.Series(
      {
        "json_valide": sum(
          {"categorie", "priorite", "reponse"} <= o.keys() for o in objets
        ),
        "categorie": sum(
          o.get("categorie") == c for o, c in zip(objets, test_part["categorie"])
        ),
        "priorite": sum(
          o.get("priorite") == p for o, p in zip(objets, test_part["priorite"])
        ),
        "charte": sum(respecte_charte(r) for r in reponses),
        "mots": sum(len(r.split()) for r in reponses),
      }
    )
    .div(len(sorties))
    .round(2)
  )


def genere_tout(
  modele, conversations: list[list[dict]], taille_lot: int = 16, max_tokens: int = 384
) -> list[str]:
  textes = []
  for debut in range(0, len(conversations), taille_lot):
    prompts = [
      tokenizer.apply_chat_template(
        c, tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for c in conversations[debut : debut + taille_lot]
    ]
    entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      sorties = modele.generate(
        **entrees,
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    textes += tokenizer.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes


def conversation_seule(message: str) -> list[dict]:
  return [
    {"role": "system", "content": CONSIGNES},
    {"role": "user", "content": message},
  ]


EXEMPLES_IDS = ["T-13559", "T-13060", "T-12282"]
exemples = tickets.loc[EXEMPLES_IDS]


def conversation_exemples(message: str) -> list[dict]:
  conversation = [{"role": "system", "content": CONSIGNES}]
  for ligne in exemples.itertuples():
    attendu = {
      "categorie": ligne.categorie,
      "priorite": ligne.priorite,
      "reponse": ligne.reponse,
    }
    conversation += [
      {"role": "user", "content": ligne.message},
      {"role": "assistant", "content": json.dumps(attendu, ensure_ascii=False)},
    ]
  return conversation + [{"role": "user", "content": message}]


def conversation_charte(message: str) -> list[dict]:
  conversation = conversation_exemples(message)
  conversation[0] = {"role": "system", "content": CONSIGNES + "\n\n" + charte}
  return conversation

## 1. Recharger le modèle affiné

L'adaptateur ne contient que les matrices de LoRA : pour s'en servir, on recharge le modèle de base comme à l'entraînement, puis on lui ajoute l'adaptateur. Écrivez `charge_modele()`, qui charge `MODELE` sur `DEVICE` en `DTYPE`, quantifié en NF4 avec double quantification sur GPU (le QLoRA de l'entraînement), tel quel sans GPU, puis lui ajoute l'adaptateur du dossier `ADAPTATEUR` et le renvoie en mode évaluation.

Puis générez avec `genere_tout` les sorties des trois approches de `APPROCHES`, ci-dessous, sur les messages de `test_part`, dans un dictionnaire `sorties` : le modèle de base avec la consigne seule, le modèle de base avec le meilleur prompt, et le modèle affiné avec la consigne seule. Le modèle de base est le même objet, l'adaptateur désactivé : inutile de le charger deux fois. Rassemblez les mesures d'`evalue` dans un tableau `comparatif`, une ligne par approche, avec le temps par ticket.

Le modèle affiné atteint-il, avec la consigne courte, la barre du meilleur prompt ? Lisez une ou deux de ses réponses : qu'est-ce que ces mesures ne disent pas ?

**Documentation utile**

- [`PeftModel.from_pretrained`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.from_pretrained)
- [`PeftModel.disable_adapter`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.disable_adapter)
- [`contextlib.nullcontext`](https://docs.python.org/3/library/contextlib.html#contextlib.nullcontext)

In [ ]:
# Pour chaque approche : l'adaptateur actif ou non, et la conversation.
APPROCHES = {
  "base, consigne seule": (False, conversation_seule),
  "base, charte et exemples": (False, conversation_charte),
  "affiné, consigne seule": (True, conversation_seule),
}
messages = list(test_part["message"])

In [ ]:
# Votre code ici

### Conseils supplémentaires

- La quantification de l'entraînement : `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)` sur GPU, `None` sans GPU
- `AutoModelForCausalLM.from_pretrained(MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=quantification)`, puis `PeftModel.from_pretrained(base, ADAPTATEUR)` : le modèle de base avec l'adaptateur, actif ; `.eval()` le met en mode évaluation (sans *dropout*)
- `with modele.disable_adapter():` : dans ce bloc, `modele` est le modèle de base ; `contextlib.nullcontext()`, un contexte qui ne change rien, pour l'approche affinée
- Sur GPU, comptez 3 à 5 minutes par approche, un peu plus pour le meilleur prompt, plus long

```python
def charge_modele() -> PeftModel:
  quantification = BitsAndBytesConfig(...) if GPU else None
  base = AutoModelForCausalLM.from_pretrained(...)
  return PeftModel.from_pretrained(...).eval()


modele = charge_modele()
sorties, secondes = {}, {}
for nom, (affine, fabrique) in APPROCHES.items():
  debut = time.time()
  with contextlib.nullcontext() if affine else ...:
    sorties[nom] = genere_tout(modele, [fabrique(m) for m in messages])
  secondes[nom] = round((time.time() - debut) / len(messages), 1)
comparatif = pd.DataFrame({nom: evalue(s) for nom, s in sorties.items()}).T
comparatif["s_par_ticket"] = pd.Series(secondes)
print(sorties["affiné, consigne seule"][0])
comparatif
```

### Solution

In [ ]:
def charge_modele() -> PeftModel:
  quantification = (
    BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_use_double_quant=True,
      bnb_4bit_compute_dtype=DTYPE,
    )
    if GPU
    else None
  )
  base = AutoModelForCausalLM.from_pretrained(
    MODELE, dtype=DTYPE, device_map=DEVICE, quantization_config=quantification
  )
  return PeftModel.from_pretrained(base, ADAPTATEUR).eval()


modele = charge_modele()
sorties, secondes = {}, {}
for nom, (affine, fabrique) in APPROCHES.items():
  debut = time.time()
  with contextlib.nullcontext() if affine else modele.disable_adapter():
    sorties[nom] = genere_tout(modele, [fabrique(m) for m in messages])
  secondes[nom] = round((time.time() - debut) / len(messages), 1)
comparatif = pd.DataFrame({nom: evalue(s) for nom, s in sorties.items()}).T
comparatif["s_par_ticket"] = pd.Series(secondes)
print(sorties["affiné, consigne seule"][0])
comparatif

# Ces mesures disent si la sortie a la forme voulue : un JSON, les bonnes
# étiquettes, les règles de forme de la charte. Pas si la réponse est juste : un
# délai de retour faux, une marche à suivre inventée passent tous ces tests.

## 2. ROUGE-L et perplexité

Deux métriques automatiques de plus, sur la réponse elle-même :

- ROUGE-L : la plus longue suite de mots, dans l'ordre, commune à la `reponse` générée et à la réponse de référence, en F-mesure, de 0 à 1. Écrivez `rouge_l(reponses)`, le ROUGE-L d'une liste de réponses contre celles de `test_part`.
- La perplexité des réponses de référence : à quel point le modèle s'attendait au JSON attendu, après le prompt de l'approche, sans rien générer. C'est l'exponentielle de la perte moyenne sur le test, celle que suit l'entraînement. Écrivez `perplexite(conversations)`, celle de `modele` tel qu'il est (l'adaptateur actif ou non).

Ajoutez à `comparatif` une colonne `rouge_l` et une colonne `perplexite`. Calculez aussi le ROUGE-L des réponses que les conseillers ont écrites à ces mêmes tickets, dans l'export (`tickets.loc[test_part["ticket_id"], "reponse"]`) : des réponses humaines, presque toutes conformes à la charte (une sur quinze vient de Jordan, l'agent qui tutoie).

Que récompense ROUGE-L, et que penser du score des conseillers ? Que mesure la perplexité, que ne mesure-t-elle pas ?

**Documentation utile**

- [La métrique ROUGE d'`evaluate`](https://huggingface.co/spaces/evaluate-metric/rouge)
- [La perplexité avec `transformers`](https://huggingface.co/docs/transformers/perplexity)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `rouge = evaluate.load("rouge")`, puis `rouge.compute(predictions=..., references=..., rouge_types=["rougeL"], tokenizer=mots)["rougeL"]` ; sans `tokenizer`, ROUGE ne garde que les lettres de a à z et les chiffres, et « réponse » devient « r ponse » : `mots(texte)` découpe en mots avec leurs accents, `re.findall(r"\w+", texte.lower())`
- La réponse générée : `str(lit_json(sortie).get("reponse", ""))`
- La perplexité d'un ticket : le prompt en texte, `tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True, enable_thinking=False)`, suivi du JSON attendu tel que le modèle l'a appris, `json.dumps({"categorie": ..., "priorite": ..., "reponse": ...}, ensure_ascii=False)`, et du token de fin de tour, `"<|im_end|>"`
- `modele(input_ids=ids, labels=etiquettes).loss` : la perte moyenne sur les tokens de `etiquettes` qui ne valent pas -100 ; mettez -100 sur ceux du prompt (`etiquettes[:, :taille_prompt] = -100`) pour ne compter que la réponse ; la perplexité est `math.exp` de la moyenne des pertes
- Sans GPU, la perplexité du meilleur prompt prend quelques minutes : 2 000 tokens de prompt par ticket

```python
rouge = evaluate.load("rouge")


def mots(texte: str) -> list[str]:
  return re.findall(r"\w+", texte.lower())


def rouge_l(reponses: list[str]) -> float:
  scores = rouge.compute(...)
  return round(float(scores["rougeL"]), 3)


def perplexite(conversations: list[list[dict]]) -> float:
  pertes = []
  for conversation, ligne in zip(conversations, test_part.itertuples()):
    prompt = tokenizer.apply_chat_template(...)
    attendu = json.dumps(...)
    taille_prompt = len(tokenizer(prompt).input_ids)
    ids = tokenizer(prompt + attendu + "<|im_end|>", return_tensors="pt").input_ids
    ids = ids.to(DEVICE)
    etiquettes = ids.clone()
    etiquettes[:, :taille_prompt] = ...
    with torch.no_grad():
      pertes.append(...)
  return round(math.exp(sum(pertes) / len(pertes)), 2)


for nom, (affine, fabrique) in APPROCHES.items():
  reponses = [str(lit_json(s).get("reponse", "")) for s in sorties[nom]]
  comparatif.loc[nom, "rouge_l"] = rouge_l(reponses)
  with ...:
    comparatif.loc[nom, "perplexite"] = perplexite([fabrique(m) for m in messages])
conseillers = list(tickets.loc[test_part["ticket_id"], "reponse"].fillna(""))
print("ROUGE-L des conseillers :", rouge_l(conseillers))
comparatif
```

### Solution

In [ ]:
rouge = evaluate.load("rouge")


def mots(texte: str) -> list[str]:
  return re.findall(r"\w+", texte.lower())


def rouge_l(reponses: list[str]) -> float:
  scores = rouge.compute(
    predictions=reponses,
    references=list(test_part["reponse"]),
    rouge_types=["rougeL"],
    tokenizer=mots,
  )
  return round(float(scores["rougeL"]), 3)


def perplexite(conversations: list[list[dict]]) -> float:
  pertes = []
  for conversation, ligne in zip(conversations, test_part.itertuples()):
    prompt = tokenizer.apply_chat_template(
      conversation, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    attendu = json.dumps(
      {
        "categorie": ligne.categorie,
        "priorite": ligne.priorite,
        "reponse": ligne.reponse,
      },
      ensure_ascii=False,
    )
    taille_prompt = len(tokenizer(prompt).input_ids)
    ids = tokenizer(prompt + attendu + "<|im_end|>", return_tensors="pt").input_ids
    ids = ids.to(DEVICE)
    etiquettes = ids.clone()
    etiquettes[:, :taille_prompt] = -100
    with torch.no_grad():
      pertes.append(modele(input_ids=ids, labels=etiquettes).loss.item())
  return round(math.exp(sum(pertes) / len(pertes)), 2)


for nom, (affine, fabrique) in APPROCHES.items():
  reponses = [str(lit_json(s).get("reponse", "")) for s in sorties[nom]]
  comparatif.loc[nom, "rouge_l"] = rouge_l(reponses)
  with contextlib.nullcontext() if affine else modele.disable_adapter():
    comparatif.loc[nom, "perplexite"] = perplexite([fabrique(m) for m in messages])
conseillers = list(tickets.loc[test_part["ticket_id"], "reponse"].fillna(""))
print("ROUGE-L des conseillers :", rouge_l(conseillers))
comparatif

# ROUGE-L compte les mots communs avec une réponse de référence parmi toutes les
# bonnes : des conseillers qui répondent juste, avec leurs mots, n'en partagent
# qu'une partie, et le modèle affiné, qui a appris les tournures de la référence,
# peut les dépasser. La perplexité dit à quel point le modèle attendait la réponse
# de référence : rien sur ce qu'il génère lui-même.

## 3. Un LLM juge

Que la réponse dise juste, le bon délai, la bonne marche à suivre, aucune règle ne le vérifie : on le confie à un modèle plus fort, le juge, qui reçoit la charte et la réponse de référence, et note chaque réponse sur une grille. Le modèle évalué est d'abord libéré, et le juge chargé à sa place (cellule ci-dessous). `juge_tout(grille, demandes)` lui fait lire `grille` en message `system`, puis chaque demande en message `user`, et renvoie ses réponses ; `tableau_verdicts(textes)` en fait un tableau, une ligne par verdict, une colonne par critère de `CRITERES`.

Écrivez `GRILLE`, le message `system` du juge : son rôle, ce qu'il reçoit, les six critères de `CRITERES`, chacun noté `true` ou `false`, le format de sa réponse, un objet JSON avec les six critères et une clé `probleme` (le principal défaut de la réponse, en quelques mots), puis la charte. Écrivez `demande_juge(ligne, sortie)`, la demande pour une ligne de `test_part` (son message, sa réponse de référence) et la sortie d'un modèle (sa catégorie et sa réponse). Faites juger les sorties des trois approches, dans un dictionnaire `verdicts`, puis rassemblez dans un tableau `notes`, une ligne par approche, la part de réponses qui passent chaque critère.

Où l'affinage a-t-il progressé, où pas ? Lisez quelques `probleme` : le juge a-t-il raison ?

In [ ]:
del modele
libere()
tokenizer_juge = AutoTokenizer.from_pretrained(JUGE)
juge = AutoModelForCausalLM.from_pretrained(JUGE, dtype=DTYPE, device_map=DEVICE)
CRITERES = ["categorie", "forme", "faits", "etape", "sans_promesse", "ton"]


def juge_tout(grille: str, demandes: list[str], max_tokens: int = 160) -> list[str]:
  """Les réponses du juge à chaque demande, `grille` en message system.

  Le début des prompts, la grille et sa charte, est le même pour toutes les
  demandes : il n'est calculé qu'une fois, et chaque génération repart d'une copie
  de son cache, comme le fait un serveur d'inférence (le cache de préfixe).
  """
  systeme = [{"role": "system", "content": grille}]
  debut = tokenizer_juge.apply_chat_template(systeme, tokenize=False)
  cache = DynamicCache(config=juge.config)
  with torch.no_grad():
    juge(**tokenizer_juge(debut, return_tensors="pt").to(DEVICE), past_key_values=cache)
  textes = []
  for demande in demandes:
    prompt = tokenizer_juge.apply_chat_template(
      [*systeme, {"role": "user", "content": demande}],
      tokenize=False,
      add_generation_prompt=True,
      enable_thinking=False,
    )
    entree = tokenizer_juge(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
      sortie = juge.generate(
        **entree,
        past_key_values=copy.deepcopy(cache),
        max_new_tokens=max_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    texte = sortie[0, entree.input_ids.shape[1] :]
    textes.append(tokenizer_juge.decode(texte, skip_special_tokens=True))
  return textes


def tableau_verdicts(textes: list[str]) -> pd.DataFrame:
  """Un verdict par ligne : un critère vaut True si le juge l'a dit, False sinon."""
  objets = [lit_json(t) for t in textes]
  tableau = pd.DataFrame({c: [o.get(c) is True for o in objets] for c in CRITERES})
  tableau["valide"] = [set(CRITERES) <= o.keys() for o in objets]
  tableau["probleme"] = [str(o.get("probleme", "")) for o in objets]
  return tableau

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les critères, à définir dans la grille :
  - `categorie` : la catégorie choisie est celle que la charte donne à ce message ;
  - `forme` : la réponse commence par la salutation de la charte, vouvoie le client, compte 40 à 120 mots et finit par la signature de la charte ;
  - `faits` : les délais, montants et conditions donnés sont exacts, selon la politique de la charte et la réponse de référence ;
  - `etape` : elle dit clairement au client ce qu'il doit faire, ou ce que nous faisons ;
  - `sans_promesse` : elle ne promet aucun remboursement avant le contrôle du produit ou la vérification du paiement ;
  - `ton` : elle est courtoise et professionnelle.
- `probleme` en premier dans le JSON : le juge écrit sa critique avant ses notes, qui en tiennent compte
- La sortie d'un modèle : `objet = lit_json(sortie)`, puis `objet.get("categorie")` et `objet.get("reponse", sortie)`, la sortie entière quand ce n'est pas du JSON
- `tableau_verdicts(textes)[CRITERES].mean()` : la part de `True` de chaque critère
- Sur GPU, comptez un quart d'heure pour les trois approches (450 verdicts) ; sans GPU, une minute par verdict environ
- Un exemple de réponse dans la grille aide un petit juge à tenir le format, mais il le recopie parfois : variez-y les valeurs

```python
GRILLE = f"""Tu évalues les réponses du service client de Biclou & Cie, magasins
de vélos. On te donne ... Juge la réponse à évaluer selon la charte ci-dessous et la
réponse de référence, critère par critère, par true ou false :
- categorie : ...
- forme : ...
- faits : ...
- etape : ...
- sans_promesse : ...
- ton : ...
Réponds uniquement par un objet JSON sur une seule ligne : la clé probleme (...),
puis les six critères, chacun true ou false. Par exemple :
{{"probleme": "...", "categorie": true, "forme": true, "faits": true, "etape": false, ...}}

{charte}"""


def demande_juge(ligne, sortie: str) -> str:
  objet = lit_json(sortie)
  return (
    f"Message du client :\n{ligne.message}\n\n"
    f"Réponse de référence :\n{ligne.reponse}\n\n"
    f"Catégorie choisie : {...}\n\n"
    f"Réponse à évaluer :\n{...}"
  )


verdicts = {}
for nom in APPROCHES:
  demandes = [
    demande_juge(ligne, s) for ligne, s in zip(test_part.itertuples(), sorties[nom])
  ]
  verdicts[nom] = tableau_verdicts(juge_tout(GRILLE, demandes))
notes = pd.DataFrame({nom: v[[*CRITERES, "valide"]].mean() for nom, v in ...}).T
print(verdicts["affiné, consigne seule"]["probleme"].head().to_list())
notes.round(2)
```

### Solution

In [ ]:
GRILLE = f"""Tu évalues les réponses du service client de Biclou & Cie, magasins
de vélos. On te donne le message d'un client, la réponse de référence validée par
l'équipe, puis la catégorie choisie et la réponse à évaluer. Juge la réponse à
évaluer selon la charte ci-dessous et la réponse de référence, critère par critère,
par true ou false :
- categorie : la catégorie choisie est celle que la charte donne à ce message ;
- forme : la réponse commence par la salutation de la charte, vouvoie le client,
  compte 40 à 120 mots et finit par la signature de la charte ;
- faits : les délais, montants et conditions qu'elle donne sont exacts, selon la
  politique de la charte et la réponse de référence ;
- etape : elle dit clairement au client ce qu'il doit faire, ou ce que nous faisons ;
- sans_promesse : elle ne promet aucun remboursement avant le contrôle du produit
  ou la vérification du paiement ;
- ton : elle est courtoise et professionnelle.
Réponds uniquement par un objet JSON sur une seule ligne : la clé probleme (le
principal défaut de la réponse, en dix mots au plus, ou "aucun"), puis les six
critères, chacun true ou false. Par exemple :
{{"probleme": "ne dit pas au client quoi faire ensuite", "categorie": true, "forme": true, "faits": true, "etape": false, "sans_promesse": true, "ton": true}}

{charte}"""


def demande_juge(ligne, sortie: str) -> str:
  objet = lit_json(sortie)
  return (
    f"Message du client :\n{ligne.message}\n\n"
    f"Réponse de référence :\n{ligne.reponse}\n\n"
    f"Catégorie choisie : {objet.get('categorie', 'aucune')}\n\n"
    f"Réponse à évaluer :\n{objet.get('reponse', sortie)}"
  )


verdicts = {}
for nom in APPROCHES:
  demandes = [
    demande_juge(ligne, s) for ligne, s in zip(test_part.itertuples(), sorties[nom])
  ]
  verdicts[nom] = tableau_verdicts(juge_tout(GRILLE, demandes))
notes = pd.DataFrame(
  {nom: v[[*CRITERES, "valide"]].mean() for nom, v in verdicts.items()}
).T
print(verdicts["affiné, consigne seule"]["probleme"].head().to_list())
notes.round(2)

## 4. Calibrer le juge

Le juge note ce qu'aucun script ne vérifie : peut-on le croire ? Deux confrontations.

1. Là où un script sait. Sur tous les verdicts de l'exercice 3, comparez le critère `categorie` du juge à la vérité (la catégorie de la sortie est celle de `test_part`), et son critère `forme` à `respecte_charte` : la part d'accord de chacun.
2. Là où seul un humain sait. Ci-dessous, 6 tickets du test (`calibration`), avec la réponse qu'un conseiller y a faite dans l'export, et l'annotation d'un collègue (`annotation`) sur les quatre autres critères. Lisez les six réponses et corrigez l'annotation dans sa cellule, là où vous n'êtes pas d'accord : trois notes au moins se discutent. Puis faites juger ces réponses (avec la catégorie du test), et calculez, critère par critère, la part d'accord du juge avec l'annotation.

Quels critères le juge tient-il ? Lesquels ne lui confieriez-vous pas, et que changeriez-vous : la grille, des exemples dans son prompt, ou le juge ?

In [ ]:
CALIBRATION_IDS = [
  "T-13207",
  "T-13215",
  "T-13238",
  "T-13379",
  "T-13429",
  "T-13782",
]
calibration = test.set_index("ticket_id").loc[CALIBRATION_IDS]
calibration["reponse_conseiller"] = tickets.loc[CALIBRATION_IDS, "reponse"]
for ligne in calibration.itertuples():
  print(f"--- {ligne.Index} ({ligne.categorie})\n{ligne.message}\n")
  print(f"Réponse du conseiller :\n{ligne.reponse_conseiller}\n")

In [ ]:
# L'annotation d'un collègue, une ligne par réponse du conseiller : corrigez-la.
annotation = pd.DataFrame(
  {
    "faits": [True, True, True, True, False, False],
    "etape": [True, True, False, True, True, False],
    "sans_promesse": [True, True, True, True, False, True],
    "ton": [True, True, True, False, False, False],
  },
  index=CALIBRATION_IDS,
)
annotation

In [ ]:
# Votre code ici

### Conseils supplémentaires

- La vérité d'un verdict sur la catégorie : `lit_json(sortie).get("categorie") == ligne.categorie` ; sur la forme : `respecte_charte(str(lit_json(sortie).get("reponse", "")))`
- `pd.Series(...) == verdicts[nom]["categorie"]` donne, ligne par ligne, l'accord ; sa moyenne, la part d'accord ; `pd.concat` met bout à bout les accords des trois approches
- La sortie à juger pour un ticket de `calibration` : `json.dumps({"categorie": ligne.categorie, "reponse": ligne.reponse_conseiller}, ensure_ascii=False)`, que `demande_juge(ligne, sortie)` lit comme une sortie de modèle
- `verdicts_calibration[annotation.columns].values == annotation.values`, puis `.mean(axis=0)` : la part d'accord par critère

```python
accords = []
for nom in APPROCHES:
  objets = [lit_json(s) for s in sorties[nom]]
  vraie_categorie = [o.get("categorie") == c for o, c in zip(objets, ...)]
  vraie_forme = [respecte_charte(str(o.get("reponse", ""))) for o in objets]
  accords.append(
    pd.DataFrame(
      {
        "categorie": verdicts[nom]["categorie"].values == vraie_categorie,
        "forme": ...,
      }
    )
  )
print("Accord avec les scripts :", pd.concat(accords).mean().round(2).to_dict())

demandes = [demande_juge(ligne, json.dumps(...)) for ligne in calibration.itertuples()]
verdicts_calibration = tableau_verdicts(juge_tout(GRILLE, demandes))
verdicts_calibration.index = CALIBRATION_IDS
accord = (verdicts_calibration[annotation.columns].values == ...).mean(axis=0)
print("Accord avec l'annotation :", dict(zip(annotation.columns, accord.round(2))))
verdicts_calibration
```

### Solution

In [ ]:
accords = []
for nom in APPROCHES:
  objets = [lit_json(s) for s in sorties[nom]]
  vraie_categorie = [
    o.get("categorie") == c for o, c in zip(objets, test_part["categorie"])
  ]
  vraie_forme = [respecte_charte(str(o.get("reponse", ""))) for o in objets]
  accords.append(
    pd.DataFrame(
      {
        "categorie": verdicts[nom]["categorie"].values == vraie_categorie,
        "forme": verdicts[nom]["forme"].values == vraie_forme,
      }
    )
  )
print("Accord avec les scripts :", pd.concat(accords).mean().round(2).to_dict())

demandes = [
  demande_juge(
    ligne,
    json.dumps(
      {"categorie": ligne.categorie, "reponse": ligne.reponse_conseiller},
      ensure_ascii=False,
    ),
  )
  for ligne in calibration.itertuples()
]
verdicts_calibration = tableau_verdicts(juge_tout(GRILLE, demandes))
verdicts_calibration.index = CALIBRATION_IDS
accord = (verdicts_calibration[annotation.columns].values == annotation.values).mean(
  axis=0
)
print("Accord avec l'annotation :", dict(zip(annotation.columns, accord.round(2))))
verdicts_calibration

## 5. Le biais de position

Plutôt que de noter chaque réponse, on peut demander au juge laquelle de deux réponses est la meilleure : c'est ainsi qu'on compare deux modèles. Mais un juge préfère souvent la réponse qu'il lit en premier, ou en second, quel que soit son contenu.

Écrivez `GRILLE_PAIRE`, le message `system` d'un juge qui compare deux réponses, A et B, selon la charte et la réponse de référence, et répond par un objet JSON `{"meilleure": "A"}` ou `{"meilleure": "B"}` ; et `demande_paire(ligne, reponse_a, reponse_b)`. Sur les tickets de `test_part.head(50)`, faites comparer la réponse du modèle de base avec le meilleur prompt et celle du modèle affiné, dans un ordre puis dans l'autre. Comptez la part des paires où le juge garde le même gagnant quand on échange les réponses, et, quand il en change, la part où il choisit la première. Que vaut une comparaison jugée dans un seul ordre ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les deux réponses : `str(lit_json(s).get("reponse", s))` pour chaque sortie de `sorties["base, charte et exemples"]` et de `sorties["affiné, consigne seule"]`, sur les `len(paires)` premiers tickets
- `juge_tout(GRILLE_PAIRE, demandes, max_tokens=16)` : un verdict court se génère vite
- Dans le premier ordre, `"A"` désigne la base ; dans le second, l'affiné. Ramenez chaque verdict au nom de son gagnant, `"base"` ou `"affiné"`, pour comparer les deux ordres
- Quand le gagnant change, le juge a choisi deux fois la même lettre : `"A"` deux fois, c'est la première position qui gagne
- La règle pour comparer deux modèles : une victoire ne compte que si les deux ordres s'accordent

```python
GRILLE_PAIRE = f"""Tu compares deux réponses du service client de Biclou & Cie, ...
Réponds uniquement par un objet JSON : {{"meilleure": "A"}} ou {{"meilleure": "B"}}.

{charte}"""


def demande_paire(ligne, reponse_a: str, reponse_b: str) -> str:
  return (
    f"Message du client :\n{ligne.message}\n\n"
    f"Réponse de référence :\n{ligne.reponse}\n\n"
    f"Réponse A :\n{...}\n\n"
    f"Réponse B :\n{...}"
  )


paires = test_part.head(50)
base = [str(lit_json(s).get("reponse", s)) for s in sorties[...][: len(paires)]]
affine = [...]
lettres = []
for premiere, seconde in [(base, affine), (affine, base)]:
  demandes = [
    demande_paire(ligne, a, b)
    for ligne, a, b in zip(paires.itertuples(), premiere, seconde)
  ]
  textes = juge_tout(GRILLE_PAIRE, demandes, max_tokens=16)
  lettres.append([lit_json(t).get("meilleure") for t in textes])
gagnants_1 = [{"A": "base", "B": "affiné"}.get(x) for x in lettres[0]]
gagnants_2 = [...]
constants = [g1 is not None and g1 == g2 for g1, g2 in zip(gagnants_1, gagnants_2)]
changes = [(x, y) for x, y, c in zip(*lettres, constants) if not c]
print(f"Même gagnant dans les deux ordres : {sum(constants) / len(paires):.0%}")
if changes:
  premiere = sum(x == y == "A" for x, y in changes) / len(changes)
  print(f"Quand le gagnant change, la première réponse gagne : {premiere:.0%}")
print(
  "Victoires dans les deux ordres :",
  {nom: sum(g == nom and c for g, c in zip(gagnants_1, constants)) for nom in ...},
)
```

### Solution

In [ ]:
GRILLE_PAIRE = f"""Tu compares deux réponses du service client de Biclou & Cie, magasins
de vélos, au même message d'un client. On te donne le message, la réponse de
référence validée par l'équipe, puis les réponses A et B. Dis laquelle suit le mieux
la charte ci-dessous et la réponse de référence : des faits exacts, une prochaine
étape claire, aucune promesse de remboursement avant contrôle, un ton courtois.
Réponds uniquement par un objet JSON : {{"meilleure": "A"}} ou {{"meilleure": "B"}}.

{charte}"""


def demande_paire(ligne, reponse_a: str, reponse_b: str) -> str:
  return (
    f"Message du client :\n{ligne.message}\n\n"
    f"Réponse de référence :\n{ligne.reponse}\n\n"
    f"Réponse A :\n{reponse_a}\n\n"
    f"Réponse B :\n{reponse_b}"
  )


paires = test_part.head(50)
base = [
  str(lit_json(s).get("reponse", s))
  for s in sorties["base, charte et exemples"][: len(paires)]
]
affine = [
  str(lit_json(s).get("reponse", s))
  for s in sorties["affiné, consigne seule"][: len(paires)]
]
lettres = []
for premiere, seconde in [(base, affine), (affine, base)]:
  demandes = [
    demande_paire(ligne, a, b)
    for ligne, a, b in zip(paires.itertuples(), premiere, seconde)
  ]
  textes = juge_tout(GRILLE_PAIRE, demandes, max_tokens=16)
  lettres.append([lit_json(t).get("meilleure") for t in textes])
gagnants_1 = [{"A": "base", "B": "affiné"}.get(x) for x in lettres[0]]
gagnants_2 = [{"A": "affiné", "B": "base"}.get(x) for x in lettres[1]]
constants = [g1 is not None and g1 == g2 for g1, g2 in zip(gagnants_1, gagnants_2)]
changes = [(x, y) for x, y, c in zip(*lettres, constants) if not c]
print(f"Même gagnant dans les deux ordres : {sum(constants) / len(paires):.0%}")
if changes:
  premiere = sum(x == y == "A" for x, y in changes) / len(changes)
  print(f"Quand le gagnant change, la première réponse gagne : {premiere:.0%}")
print(
  "Victoires dans les deux ordres :",
  {
    nom: sum(g == nom and c for g, c in zip(gagnants_1, constants))
    for nom in ["base", "affiné"]
  },
)

# Une paire jugée dans un seul ordre mêle la qualité des réponses et la place
# qu'elles occupent : les victoires qui ne tiennent pas dans les deux ordres ne
# disent rien des modèles.

## 6. Non-régression : un benchmark

En apprenant les tickets, le modèle a-t-il perdu ce qu'il savait faire ? Un benchmark public le mesure avant et après, de la même façon pour tous : [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness), d'EleutherAI, en rassemble des centaines. On prend Belebele (Meta, 2023), des questions de compréhension à choix multiples (A à D) sur des passages de Wikipédia, 900 par langue : sa tâche française est `belebele_fra_Latn`. Le harnais n'y génère rien : il compare la vraisemblance de chaque lettre après la question.

Le juge est libéré, et le modèle évalué rechargé (cellule ci-dessous). Mesurez l'exactitude du modèle de base, puis du modèle affiné, sur les `LIMITE` premières questions : 200 sur GPU, 8 sans GPU, où le chiffre ne dit rien et seul le code compte. L'écart entre les deux modèles dépasse-t-il l'erreur type de la mesure ?

**Documentation utile**

- [L'API Python du harnais](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/python-api.md)
- [La tâche Belebele](https://github.com/EleutherAI/lm-evaluation-harness/tree/main/lm_eval/tasks/belebele)

In [ ]:
del juge
libere()
modele = charge_modele()
LIMITE = 200 if GPU else 8

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `HFLM(pretrained=modele, tokenizer=tokenizer, batch_size=8)` : le harnais sur un modèle déjà chargé, l'adaptateur compris ; pour un modèle à charger depuis le Hub, `simple_evaluate(model="hf", model_args="pretrained=Qwen/Qwen3-1.7B,peft=<dossier>")` ferait de même
- `simple_evaluate(model=harnais, tasks=["belebele_fra_Latn"], limit=LIMITE)` renvoie un dictionnaire : l'exactitude dans `resultats["results"]["belebele_fra_Latn"]["acc,none"]`, son erreur type dans `["acc_stderr,none"]`
- Le modèle de base : le même appel, sous `with modele.disable_adapter():`
- Sur GPU, une à deux minutes par modèle ; les lignes `fatal : ... dépôt git` (`fatal: not a git repository`) viennent du harnais, qui cherche la version de son code : elles sont sans effet

```python
harnais = HFLM(pretrained=modele, tokenizer=tokenizer, batch_size=8)
benchmark = {}
for nom, affine in [("base", False), ("affiné", True)]:
  with contextlib.nullcontext() if affine else ...:
    resultats = simple_evaluate(...)
  mesure = resultats["results"]["belebele_fra_Latn"]
  benchmark[nom] = {"exactitude": ..., "erreur_type": ...}
pd.DataFrame(benchmark).T.round(3)
```

### Solution

In [ ]:
harnais = HFLM(pretrained=modele, tokenizer=tokenizer, batch_size=8)
benchmark = {}
for nom, affine in [("base", False), ("affiné", True)]:
  with contextlib.nullcontext() if affine else modele.disable_adapter():
    resultats = simple_evaluate(
      model=harnais, tasks=["belebele_fra_Latn"], limit=LIMITE
    )
  mesure = resultats["results"]["belebele_fra_Latn"]
  benchmark[nom] = {
    "exactitude": mesure["acc,none"],
    "erreur_type": mesure["acc_stderr,none"],
  }
pd.DataFrame(benchmark).T.round(3)

## 7. Non-régression : hors du service client

Un benchmark à choix multiples ne voit pas ce qu'on remarque le plus vite chez un modèle affiné : il applique sa tâche partout. Posez au modèle de base et au modèle affiné les questions générales de `QUESTIONS`, ci-dessous, avec un autre message `system`, `ASSISTANT`, puis comptez pour chacun la part de bonnes réponses (le mot attendu est dans la réponse) et la part de réponses qui sont un ticket en JSON (`lit_json` y trouve une `categorie`). Lisez les réponses : que reste-t-il du modèle de départ ? Quels remèdes du cours essaieriez-vous ?

In [ ]:
ASSISTANT = "Tu es un assistant généraliste. Réponds en une ou deux phrases."
# Chaque question, et un mot que sa bonne réponse contient.
QUESTIONS = {
  "Quelle est la capitale de l'Australie ?": "Canberra",
  "Combien font 17 fois 3 ?": "51",
  "Qui a écrit Les Misérables ?": "Hugo",
  "Combien de jours compte une année bissextile ?": "366",
  "Traduis en anglais : « Le chat dort sur le canapé. »": "cat",
  "Quel gaz les plantes absorbent-elles pour la photosynthèse ?": "carbone",
}

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Une conversation : `[{"role": "system", "content": ASSISTANT}, {"role": "user", "content": question}]`
- `genere_tout(modele, conversations, max_tokens=128)` : une réponse courte suffit
- Un mot attendu dans une réponse, sans tenir compte de la casse : `attendu.lower() in reponse.lower()`
- Un ticket en JSON : `"categorie" in lit_json(reponse)`

```python
conversations = [
  [{"role": "system", "content": ASSISTANT}, {"role": "user", "content": q}]
  for q in QUESTIONS
]
generales, mesures = {}, {}
for nom, affine in [("base", False), ("affiné", True)]:
  with ...:
    generales[nom] = genere_tout(modele, conversations, max_tokens=128)
  mesures[nom] = {
    "bonnes": sum(... for r, a in zip(generales[nom], QUESTIONS.values())),
    "tickets_json": sum(... for r in generales[nom]),
  }
for question, base, affine in zip(QUESTIONS, *generales.values()):
  print(f"--- {question}\nBase : {base}\nAffiné : {affine}\n")
pd.DataFrame(mesures).T / len(QUESTIONS)
```

### Solution

In [ ]:
conversations = [
  [{"role": "system", "content": ASSISTANT}, {"role": "user", "content": q}]
  for q in QUESTIONS
]
generales, mesures = {}, {}
for nom, affine in [("base", False), ("affiné", True)]:
  with contextlib.nullcontext() if affine else modele.disable_adapter():
    generales[nom] = genere_tout(modele, conversations, max_tokens=128)
  mesures[nom] = {
    "bonnes": sum(
      a.lower() in r.lower() for r, a in zip(generales[nom], QUESTIONS.values())
    ),
    "tickets_json": sum("categorie" in lit_json(r) for r in generales[nom]),
  }
for question, base, affine in zip(QUESTIONS, *generales.values()):
  print(f"--- {question}\nBase : {base}\nAffiné : {affine}\n")
pd.DataFrame(mesures).T / len(QUESTIONS)

# Un modèle affiné sur une seule tâche, toujours avec la même consigne, peut
# répondre par un ticket à tout message : il a appris le format plus que la
# consigne. Remèdes : moins d'époques, un taux plus faible, des exemples
# généraux mêlés aux données ; ou ne servir l'adaptateur qu'aux tickets.

## 8. L'analyse d'erreurs

Les moyennes disent combien le modèle affiné se trompe ; la lecture des échecs dit pourquoi, et donc quoi changer. Réunissez dans un tableau `erreurs`, pour chaque ticket de `test_part`, la catégorie et la priorité attendues, celles du modèle affiné, `respecte_charte` de sa réponse, et les verdicts du juge (`verdicts["affiné, consigne seule"]`, `probleme` compris). Puis :

- la matrice de confusion des catégories, attendues en lignes, prédites en colonnes ;
- par catégorie attendue, la part de bonnes catégories et de chaque critère du juge ;
- les tickets en échec, ceux où la catégorie, la priorité, la forme ou un critère du juge est en défaut : lisez-en quelques-uns, le message, la réponse de référence, celle du modèle et le `probleme` du juge.

Rangez chaque échec lu selon sa cause, avec le tableau du cours : des étiquettes fausses, un cas rare, un sous-apprentissage, un surapprentissage, un fait que le modèle ne peut pas savoir. Faut-il itérer sur les données, ou sur l'entraînement ?

**Documentation utile**

- [`pd.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `objets = [lit_json(s) for s in sorties["affiné, consigne seule"]]`, puis `[o.get("categorie") for o in objets]`…
- `pd.concat([colonnes, verdicts["affiné, consigne seule"]], axis=1)` : les colonnes du ticket et celles du juge côte à côte (même index, 0 à n - 1)
- `pd.crosstab(erreurs["attendue"], erreurs["predite"])` : la matrice de confusion ; `erreurs.groupby("attendue")[colonnes].mean()`, les parts par catégorie
- Un échec : `~erreurs[colonnes_booleennes].all(axis=1)`

```python
objets = [lit_json(s) for s in sorties["affiné, consigne seule"]]
colonnes = pd.DataFrame(
  {
    "ticket_id": test_part["ticket_id"].values,
    "attendue": test_part["categorie"].values,
    "predite": [o.get("categorie") for o in objets],
    "priorite_attendue": ...,
    "priorite_predite": ...,
    "regles": [respecte_charte(str(o.get("reponse", ""))) for o in objets],
  }
)
erreurs = pd.concat([colonnes, verdicts["affiné, consigne seule"]], axis=1)
erreurs["bonne_categorie"] = erreurs["attendue"] == erreurs["predite"]
erreurs["bonne_priorite"] = ...
print(pd.crosstab(...))
print(erreurs.groupby("attendue")[["bonne_categorie", *CRITERES]].mean().round(2))
booleens = ["bonne_categorie", "bonne_priorite", "regles", *CRITERES]
echecs = erreurs[~erreurs[booleens].all(axis=1)]
print(len(echecs), "tickets en échec sur", len(erreurs))
for i in echecs.index[:5]:
  print(f"--- {erreurs.loc[i, 'ticket_id']} : {erreurs.loc[i, 'probleme']}")
  print("Message :", test_part["message"].iloc[i])
  print("Référence :", test_part["reponse"].iloc[i])
  print("Modèle :", ..., "\n")
```

### Solution

In [ ]:
objets = [lit_json(s) for s in sorties["affiné, consigne seule"]]
colonnes = pd.DataFrame(
  {
    "ticket_id": test_part["ticket_id"].values,
    "attendue": test_part["categorie"].values,
    "predite": [o.get("categorie") for o in objets],
    "priorite_attendue": test_part["priorite"].values,
    "priorite_predite": [o.get("priorite") for o in objets],
    "regles": [respecte_charte(str(o.get("reponse", ""))) for o in objets],
  }
)
erreurs = pd.concat([colonnes, verdicts["affiné, consigne seule"]], axis=1)
erreurs["bonne_categorie"] = erreurs["attendue"] == erreurs["predite"]
erreurs["bonne_priorite"] = erreurs["priorite_attendue"] == erreurs["priorite_predite"]
print(pd.crosstab(erreurs["attendue"], erreurs["predite"]))
print(erreurs.groupby("attendue")[["bonne_categorie", *CRITERES]].mean().round(2))
booleens = ["bonne_categorie", "bonne_priorite", "regles", *CRITERES]
echecs = erreurs[~erreurs[booleens].all(axis=1)]
print(len(echecs), "tickets en échec sur", len(erreurs))
for i in echecs.index[:5]:
  print(f"--- {erreurs.loc[i, 'ticket_id']} : {erreurs.loc[i, 'probleme']}")
  print("Message :", test_part["message"].iloc[i])
  print("Référence :", test_part["reponse"].iloc[i])
  print(
    "Modèle :", objets[i].get("reponse", sorties["affiné, consigne seule"][i]), "\n"
  )

# Des erreurs dispersées sur toutes les catégories, avec un format mal tenu,
# demandent plus d'entraînement ; des erreurs groupées sur une catégorie ou un cas
# rare, plus d'exemples de ce cas ; un « échec » où le modèle a raison contre le
# test, une étiquette à corriger. Et un délai ou un prix faux que les données
# n'apprennent pas : un RAG sur la charte, plutôt que l'affinage.

## À retenir

- Les métriques automatiques mesurent la forme : un JSON valide, les étiquettes, les règles vérifiables, les mots de la référence (ROUGE-L), la vraisemblance de la référence (perplexité). Aucune ne dit si la réponse est juste.
- Un LLM juge note ce qu'elles ne voient pas, sur une grille de critères notés oui ou non ; il se calibre là où un script sait, et contre une annotation humaine, avant qu'on croie ses chiffres. Comparées par paires, deux réponses se jugent dans les deux ordres.
- L'affinage peut coûter ailleurs : un benchmark général avant et après, et des questions hors sujet qui montrent si le modèle impose sa tâche partout.
- Les échecs, lus un à un et rangés par cause, disent sur quoi itérer : les données, l'entraînement, ou un autre outil que l'affinage.